# 데이터구조 05주차 실습: 데크(Deque)와 우선순위 큐(Priority Queue)

**주제:** `deque` 활용, `heapq` 기반 우선순위 큐 구현  
**학습 목표**
- 양쪽 끝에서 삽입·삭제 가능한 Deque ADT를 활용한다.
- `collections.deque`의 핵심 연산과 시간복잡도를 이해한다.
- 최소 힙(min-heap)을 이용해 우선순위 큐를 구현한다.
- 회문 검사와 작업 스케줄링 문제에 적절한 자료구조를 적용한다.

> 실행 환경: Python 3.8+ / Jupyter Notebook 또는 Google Colab


## 0. 준비
- `deque`: 양 끝 삽입·삭제가 O(1)
- `heapq`: Python 표준 라이브러리의 **최소 힙** 구현

아래 TODO를 채우고 모든 테스트를 통과시키세요.

In [1]:
from collections import deque
import heapq


def expect_error(error_type, func):
    try:
        func()
    except error_type:
        print(f"OK: {error_type.__name__} 발생")
    else:
        raise AssertionError(f"{error_type.__name__}가 발생해야 합니다.")

## 1. Deque ADT
Deque(Double-Ended Queue)는 앞과 뒤 양쪽에서 원소를 넣고 뺄 수 있습니다.

구현할 메서드: `add_front`, `add_rear`, `remove_front`, `remove_rear`, `is_empty`, `size`

In [2]:
class Deque:
    def __init__(self):
        self.items = deque()

    def add_front(self, item):
        self.items.appendleft(item)

    def add_rear(self, item):
        self.items.append(item)

    def remove_front(self):
        if self.is_empty():
            raise IndexError("remove_front from empty deque")
        return self.items.popleft()

    def remove_rear(self):
        if self.is_empty():
            raise IndexError("remove_rear from empty deque")
        return self.items.pop()

    def is_empty(self):
        return len(self.items) == 0

    def size(self):
        return len(self.items)

In [3]:
d = Deque()
d.add_rear('B'); d.add_front('A'); d.add_rear('C')
assert d.size() == 3
assert d.remove_front() == 'A'
assert d.remove_rear() == 'C'
assert d.remove_front() == 'B'
assert d.is_empty() is True
expect_error(IndexError, d.remove_front)
print('Deque 기본 테스트 통과!')

OK: IndexError 발생
Deque 기본 테스트 통과!


### 연습 1 — 회문(Palindrome) 검사
앞과 뒤에서 한 글자씩 꺼내 비교합니다. 대소문자와 공백은 무시합니다.

예: `"Never odd or even"` → `True`

In [4]:
def is_palindrome(text):
    d = Deque()
    for ch in text:
        if ch.isalnum():
            d.add_rear(ch.lower())

    while d.size() > 1:
        if d.remove_front() != d.remove_rear():
            return False
    return True

assert is_palindrome('level') is True
assert is_palindrome('Never odd or even') is True
assert is_palindrome('data structure') is False
print('회문 검사 테스트 통과!')

회문 검사 테스트 통과!


## 2. 우선순위 큐(Priority Queue)
일반 큐는 **도착 순서(FIFO)**, 우선순위 큐는 **우선순위 순서**로 작업을 꺼냅니다.

`heapq`는 최소 힙입니다. 숫자가 작을수록 먼저 나옵니다. 동점일 때 입력 순서를 보존하기 위해 순번(sequence)을 함께 저장합니다.

In [5]:
class PriorityQueue:
    def __init__(self):
        self.heap = []
        self.sequence = 0

    def enqueue(self, item, priority):
        heapq.heappush(self.heap, (priority, self.sequence, item))
        self.sequence += 1

    def dequeue(self):
        if self.is_empty():
            raise IndexError("dequeue from empty priority queue")
        return heapq.heappop(self.heap)[2]

    def is_empty(self):
        return len(self.heap) == 0

    def size(self):
        return len(self.heap)

In [6]:
pq = PriorityQueue()
pq.enqueue('일반 문의', 3)
pq.enqueue('서버 장애', 1)
pq.enqueue('비밀번호 초기화', 2)
pq.enqueue('다른 서버 장애', 1)
assert [pq.dequeue(), pq.dequeue(), pq.dequeue(), pq.dequeue()] == [
    '서버 장애', '다른 서버 장애', '비밀번호 초기화', '일반 문의'
]
expect_error(IndexError, pq.dequeue)
print('PriorityQueue 기본 테스트 통과!')

OK: IndexError 발생
PriorityQueue 기본 테스트 통과!


### 연습 2 — 작업 스케줄링
작업을 `(이름, 우선순위)` 튜플로 받아 우선순위가 높은 순서(숫자가 작은 순서)로 이름 목록을 반환하세요.

In [7]:
def schedule_jobs(jobs):
    pq = PriorityQueue()
    for name, priority in jobs:
        pq.enqueue(name, priority)

    order = []
    while not pq.is_empty():
        order.append(pq.dequeue())
    return order

jobs = [('문서 출력', 3), ('긴급 백업', 1), ('메일 발송', 2), ('장애 알림', 1)]
assert schedule_jobs(jobs) == ['긴급 백업', '장애 알림', '메일 발송', '문서 출력']
print('작업 스케줄링 테스트 통과!')

작업 스케줄링 테스트 통과!


## 3. 도전 — 상위 K개 값
`heapq.nlargest(k, values)` 또는 크기 k의 최소 힙을 이용해 상위 k개 값을 구할 수 있습니다.

아래 함수는 큰 값부터 정렬된 결과를 반환해야 합니다.

In [8]:
def top_k(values, k):
    if k <= 0:
        return []
    heap = []  # 크기 k의 최소 힙: 루트가 현재 상위 k개 중 가장 작은 값
    for v in values:
        if len(heap) < k:
            heapq.heappush(heap, v)
        elif v > heap[0]:
            heapq.heapreplace(heap, v)
    return sorted(heap, reverse=True)

assert top_k([7, 1, 9, 3, 9, 2], 3) == [9, 9, 7]
print('상위 K개 테스트 통과!')

상위 K개 테스트 통과!


## 제출 체크

|자료구조|핵심 연산|시간복잡도|
|---|---|---|
|Deque|양쪽 삽입/삭제|O(1)|
|Min-Heap|삽입·최솟값 삭제|O(log n)|
|Heapify|리스트를 힙으로 변환|O(n)|

1. 모든 TODO와 `assert`를 완료했는가?
2. 일반 Queue와 Priority Queue의 제거 기준을 설명할 수 있는가?
- 일반 Queue: 들어온 순서(FIFO)를 기준으로 제거
  가장 먼저 삽입된 원소가 가장 먼저 제거됌
- Priority Queue: 우선순위가 가장 높은 원소를 먼저 제거 
  우선순위가 같으면 함께 저장한 순번이 작은 원소
3. Deque와 Heap을 선택한 이유를 각 문제에 대해 설명할 수 있는가?
- 회문 검사 → Deque: 회문은 문자열의 양 끝 문자를 동시에 비교해야함 Deque는 remove_front와 remove_rear를 모두 O(1)에 수행하므로 양 끝에서 한 글자씩 꺼내 비교하기에 적합하고 전체 검사는 O(n)
리스트의 pop(0)은 O(n)이라 비효율적임
- 작업 스케줄링 → Heap(우선순위 큐): 매번 가장 우선순위가 높은 작업을 꺼내야힘. 최소 힙은 삽입과 최솟값 삭제가 모두 O(log n)이라, 매번 리스트 전체에서 최솟값을 찾는 방식(O(n))보다 효율적 
n개 작업 전체는 O(n log n)
- 상위 K개 → 크기 k의 최소 힙: 힙의 루트에 현재 상위 k개 중 가장 작은 값이 있으므로, 새 값이 루트보다 클 때만 heapreplace로 교체하면 됩니다. 힙 크기가 k로 유지되므로 시간복잡도는 O(n log k)이고, 전체 정렬(O(n log n))보다 효율적
4. 파일명을 `Week05_학번_이름.ipynb`로 하여 제출한다.
